# Pandas GroupBy

En pandas se puede agrupar con el metodo ".groupby()", el cual permite analizar analisar y transformar datasets cuando se trabaja con python.

Sirve para separar DataFrame en grupos basados en datos de columnas, aplicar funciones a cada grupo y combinar los resultados en un nuevo dataframe. Esta texnica es esencial en tareas como agregacion, filtrado y transformacion en datos agrupados.

## Ejemplo 1: U.S. Congress Dataset

Este código carga un dataset de legisladores históricos del Congreso de EE.UU. desde un CSV usando pd.read_csv(), aplicando tres optimizaciones clave: 
- **Primero:** define un diccionario dtypes que convierte columnas de baja cardinalidad (first_name, gender, type, state, party — es decir, columnas con pocos valores únicos que se repiten muchas veces) al tipo "category" en vez de dejarlas como object genérico, lo cual reduce el uso de memoria porque pandas almacena internamente solo los valores únicos una vez y cada fila guarda apenas un código numérico que apunta a esa categoría. 
- **Segundo:** usa usecols=list(dtypes) + ["birthday", "last_name"] para cargar únicamente las columnas necesarias del CSV (ignorando el resto), ahorrando memoria y tiempo de carga.
- **Tercero:** usa parse_dates=["birthday"] para convertir automáticamente esa columna a tipo datetime en el momento de la lectura, en vez de tener que convertirla después. Esta práctica es especialmente útil de cara a usar groupby(), ya que las operaciones de agrupación sobre columnas categóricas son más eficientes que sobre columnas de texto genérico.

In [2]:
import pandas as pd

dtypes = {
    "first_name": "category",
    "gender": "category",
    "type": "category",
    "state": "category",
    "party": "category",
}
df = pd.read_csv(
    "data/data_groupby/legislators-historical.csv",
    dtype=dtypes,
    usecols=list(dtypes) + ["birthday", "last_name"],
    parse_dates=["birthday"]
)
df.head()

,last_name,first_name,birthday,gender,type,state,party
0,Bassett,Richard,1745-04-02,M,sen,DE,Anti-Administration
1,Bland,Theodorick,1742-03-21,M,rep,VA,NaN
2,Burke,Aedanus,1743-06-16,M,rep,SC,NaN
3,Carroll,Daniel,1730-07-22,M,rep,MD,NaN
4,Clymer,George,1739-03-16,M,rep,PA,NaN


In [3]:
df.dtypes

last_name                str
first_name          category
birthday      datetime64[us]
gender              category
type                category
state               category
party               category
dtype: object

### Como usar GroupBy

Se llama a "groupby()" y se entrega el nombre de la columna con la cual se quiere agrupar que es este caso es "state". Luego,se usa "["last_name"] para especificar la colimna con la que queremos realizar la agrupacion.

In [4]:
n_by_state = df.groupby("state")["last_name"].count()
n_by_state.head()

state
AK     16
AL    206
AR    117
AS      2
AZ     48
Name: last_name, dtype: int64

Se puede entregar mas que una sola columna como primer argumento, se puede:
- Una lista con multiples columnas
- Un diccionario o pandas series
- Un Numpy Array o pandas index, o un array iterable.

In [5]:
df.groupby(["state", "gender"])["last_name"].count()

state  gender
AK     M          16
AL     F           3
       M         203
AR     F           5
       M         112
                ... 
WI     M         196
WV     F           1
       M         119
WY     F           2
       M          38
Name: last_name, Length: 104, dtype: int64

### Pandas Groupby vs SQL

En el resultado de SQL este contiene 3 columnas: 
1) state
2) gender
3) count

En la version de pandas, las columnas agrupadas son empujadas en la "MultiIndex" del resultado "series" por defecto.

In [6]:
n_by_state_gender = df.groupby(["state", "gender"])["last_name"].count()
type(n_by_state_gender)

pandas.Series

In [7]:
n_by_state_gender.index[:5]

MultiIndex([('AK', 'M'),
            ('AL', 'F'),
            ('AL', 'M'),
            ('AR', 'F'),
            ('AR', 'M')],
           names=['state', 'gender'])

Para emular mas el resultado de SQL y empujar las columnas agrupadas en columnas en el resultado, se puede usar "as_index=False".

In [8]:
df.groupby(["state", "gender"], as_index=False)["last_name"].size()  # size se parece a count pero no exluye los NaN

,state,gender,size
0,AK,M,16
1,AL,F,3
2,AL,M,203
3,AR,F,5
4,AR,M,112
...,...,...,...
99,WI,M,196
100,WV,F,1
101,WV,M,119
102,WY,F,2


Tambien se nota que las consultas SQL usan explicitamente "order by", mientras que "groupby()" no. Eso es por que con "groupby()" hace esto por defecto por el parametro "sort", que siempre es "True" mientras no se le indique lo contrario.

In [9]:
df.groupby("state", sort=False)["last_name"].count()  # No ordena los resultados por la palabra clave

state
DE      97
VA     432
SC     251
MD     305
PA    1053
MA     426
NJ     359
GA     309
NY    1461
NC     354
CT     240
VT     115
KY     373
RI     107
NH     181
TN     299
OH     674
MS     155
OL       2
IN     341
LA     197
IL     486
MO     333
AL     206
AR     117
ME     175
FL     155
MI     294
IA     202
WI     196
TX     256
CA     361
OR      89
MN     160
NM      54
NE     127
WA      95
KS     141
UT      53
NV      56
CO      90
WV     120
DK       9
AZ      48
ID      59
MT      52
WY      40
DC       2
ND      44
SD      51
OK      92
HI      23
PR      19
AK      16
PI      13
VI       4
GU       4
AS       2
Name: last_name, dtype: int64

In [10]:
by_state = df.groupby("state")

split-apply-combine refiere a una cadena de 3 pasos: 
1) Split: separa una tabla en grupos.
2) Apply: aplica algunas operaciones a cada una de las tablas pequeñas.
3) combine: combina los resultados.

La mejor manera de ver el paso de "Split" es:

In [11]:
for state, frame in by_state:
    print(f"First 2 entries for {state!r}")
    print("------------------------")
    print(frame.head(2), end="\n\n")

First 2 entries for 'AK'
------------------------
     last_name first_name   birthday gender type state        party
6619    Waskey      Frank 1875-04-20      M  rep    AK     Democrat
6647      Cale     Thomas 1848-09-17      M  rep    AK  Independent

First 2 entries for 'AL'
------------------------
    last_name first_name   birthday gender type state       party
912   Crowell       John 1780-09-18      M  rep    AL  Republican
991    Walker       John 1783-08-12      M  sen    AL  Republican

First 2 entries for 'AR'
------------------------
     last_name first_name   birthday gender type state party
1001     Bates      James 1788-08-25      M  rep    AR   NaN
1279    Conway      Henry 1793-03-18      M  rep    AR   NaN

First 2 entries for 'AS'
------------------------
          last_name first_name   birthday gender type state     party
10797         Sunia       Fofó 1937-03-13      M  rep    AS  Democrat
11755  Faleomavaega        Eni 1943-08-15      M  rep    AS  Democrat

F

Tambien se puede utilizar un atributo de "groupby()" el cual es ".groups() que muestra un diccionario de los grupos que se formaron {nombre de grupo : etiqueta de grupo}

En el ejemplo se busca la llave "PA" para ver el resultado que muestr.

In [12]:
by_state.groups['PA']

Index([    4,    19,    21,    27,    38,    57,    69,    76,    84,    88,
       ...
       11842, 11866, 11875, 11877, 11887, 11891, 11932, 11945, 11959, 11973],
      dtype='int64', length=1053)

Una forma de saber todas las llaves que contiene el diccionario del grupo creado.

In [13]:
by_state.groups.keys()

dict_keys(['AK', 'AL', 'AR', 'AS', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'DK', 'FL', 'GA', 'GU', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OL', 'OR', 'PA', 'PI', 'PR', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VI', 'VT', 'WA', 'WI', 'WV', 'WY'])

Tambien se puede usar ".get_group()" para ver una sub tabla de un solo grupo, es equivalente a ".iloc[]" ya que se puede obtener el mismo resultado con: 
- df.loc[df['state'] == 'PA']

In [14]:
by_state.get_group("PA")

,last_name,first_name,birthday,gender,type,state,party
4,Clymer,George,1739-03-16,M,rep,PA,NaN
19,Maclay,William,1737-07-20,M,sen,PA,Anti-Administration
21,Morris,Robert,1734-01-20,M,sen,PA,Pro-Administration
27,Wynkoop,Henry,1737-03-02,M,rep,PA,NaN
38,Jacobs,Israel,1726-06-09,M,rep,PA,NaN
...,...,...,...,...,...,...,...
11891,Brady,Robert,1945-04-07,M,rep,PA,Democrat
11932,Shuster,Bill,1961-01-10,M,rep,PA,Republican
11945,Rothfus,Keith,1962-04-25,M,rep,PA,Republican
11959,Costello,Ryan,1976-09-07,M,rep,PA,Republican


Continuando con la segunda parte "Apply" en la cual se aplican algunas operaciones o llamadas a cada una de las sun tablas que la parte de "Spliting" creo.

In [15]:
state, frame = next(iter(by_state))
state

'AK'

In [16]:
frame.head(3)

,last_name,first_name,birthday,gender,type,state,party
6619,Waskey,Frank,1875-04-20,M,rep,AK,Democrat
6647,Cale,Thomas,1848-09-17,M,rep,AK,Independent
7442,Grigsby,George,1874-12-02,M,rep,AK,NaN


## Example 2: Air Quality Dataset

1) pd.read_csv(..., na_values=[-200]) — carga el CSV normal, sin intentar combinar columnas de fecha automáticamente. El parámetro na_values=[-200] sigue funcionando igual: le dice a pandas que trate el valor -200 como un dato faltante (NaN), ya que este dataset de calidad del aire usa -200 como código de "sensor sin datos".
2) df["Date"] + " " + df["Time"] — concatena el texto de ambas columnas con un espacio en medio (ej: "10/03/2004" + " " + "18:00:00" → "10/03/2004 18:00:00").
3) pd.to_datetime(...) — convierte ese texto combinado a un objeto datetime real.

In [49]:
df = pd.read_csv(
    "data/data_groupby/airqual.csv",
    na_values=[-200],
    usecols=["Date", "Time", "CO(GT)", "T", "RH", "AH"]
)

# Combina las columnas Date y Time en una sola columna datetime
df["Date_Time"] = pd.to_datetime(
    df["Date"] + " " + df["Time"], 
    dayfirst=True 
)

df = df.drop(columns=["Date", "Time"])

df = df.rename(
    columns={
        "CO(GT)": "co",
        "Date_Time": "tstamp",
        "T": "temp_c",
        "RH": "rel_hum",
        "AH": "abs_hum",
    }
).set_index("tstamp")

df.head()

C:\Users\nicol\AppData\Local\Temp\ipykernel_38660\3632527982.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date_Time"] = pd.to_datetime(


,co,temp_c,rel_hum,abs_hum
tstamp,,,,
2004-10-03 18:00:00,2.6,13.6,48.9,0.7578
2004-10-03 19:00:00,2.0,13.3,47.7,0.7255
2004-10-03 20:00:00,2.2,11.9,54.0,0.7502
2004-10-03 21:00:00,2.2,11.0,60.0,0.7867
2004-10-03 22:00:00,1.6,11.2,59.6,0.7888


"co" es la lectura promedio de monóxido de carbono de esa hora, mientras que "temp_c, rel_hum y abs_hum" son la temperatura promedio en grados Celsius, la humedad relativa y la humedad absoluta durante esa hora, respectivamente. Las observaciones abarcan desde marzo de 2004 hasta abril de 2005:

In [18]:
print('Fecha mas antigua: ',df.index.min())
print('Fecha mas nueva  : ',df.index.max())

Fecha mas antigua:  2004-01-04 00:00:00
Fecha mas nueva  :  2005-12-03 23:00:00


### Grouping on Derived Arrays

Se tomara ventaja de los parametros que acepta "groupby()" de pandas en especifico el que acepta array de numpy, index de pandas y un array de iterables.

In [19]:
day_names = df.index.day_name()
type(day_names)

pandas.Index

In [20]:
day_names[:10]

Index(['Sunday', 'Sunday', 'Sunday', 'Sunday', 'Sunday', 'Sunday', 'Wednesday',
       'Wednesday', 'Wednesday', 'Wednesday'],
      dtype='str', name='tstamp')

day_dames es un array de una dimension por lo que se puede utilizar como paretro en "groupby()"

In [21]:
df.groupby(day_names)['co'].mean()

tstamp
Friday       2.435269
Monday       2.067529
Saturday     1.951165
Sunday       1.669290
Thursday     2.273714
Tuesday      2.327932
Wednesday    2.378685
Name: co, dtype: float64

Se puede ser mas especifico y aparte del dia de la semana, por la hora del dia.

In [22]:
hr = df.index.hour

df.groupby([day_names, hr])['co'].mean().rename_axis(['dow', 'hr'])

dow        hr
Friday     0     1.822000
           1     1.487755
           2     1.130612
           3     0.883673
           4     0.767647
                   ...   
Wednesday  19    4.414583
           20    3.948936
           21    2.782979
           22    2.097872
           23    1.940426
Name: co, Length: 168, dtype: float64

Crea tres categorías de temperatura usando **pd.cut(df['temp_c'], bins=3, labels=('cool', 'warm', 'hot'))**, que divide el rango completo de temperaturas en tres intervalos de igual amplitud numérica (no igual cantidad de observaciones) y les asigna las etiquetas 'cool', 'warm' y 'hot' según en qué tercio del rango caiga cada valor. Luego, **df[['rel_hum', 'abs_hum']].groupby(bins).agg(['mean', 'median'])** selecciona solo las columnas de humedad relativa y absoluta, las agrupa según esas tres categorías de temperatura (usando bins como criterio de agrupación en vez de una columna existente del DataFrame), y calcula tanto la media como la mediana de cada columna de humedad dentro de cada grupo de temperatura — permitiendo comparar, por ejemplo, si los días "hot" tienden a tener mayor o menor humedad que los días "cool".

In [23]:
bins = pd.cut(df['temp_c'], bins=3, labels=('cool', 'warm', 'hot'))
df[['rel_hum', 'abs_hum']].groupby(bins).agg(['mean', 'median'])

rel_hum          abs_hum        
             mean median      mean  median
temp_c                                    
cool    57.651452   59.2  0.665874  0.6581
warm    49.382716   49.3  1.182894  1.1452
hot     24.994334   24.1  1.292958  1.2742

### Resampling

Esta es una manera de agrupar por año y cuartil, para realizar observaciones de monoxido de carbono.

In [24]:
df.groupby([df.index.year, df.index.quarter])['co'].agg(['max', 'min']).rename_axis(['year', 'quarter'])

max  min
year quarter           
2004 1         9.4  0.1
     2         8.7  0.1
     3         7.5  0.1
     4        11.9  0.1
2005 1         8.7  0.1
     2         6.1  0.4
     3         6.0  0.4
     4         8.4  0.1

El mismo codigo de arriba se puede escribir de otra manera con "resampling", entregando solo el texto de frecuencia, como "QE" para "cuartil".

In [25]:
df.resample('QE')['co'].agg(['min', 'max'])

,min,max
tstamp,,
2004-03-31,0.1,9.4
2004-06-30,0.1,8.7
2004-09-30,0.1,7.5
2004-12-31,0.1,11.9
2005-03-31,0.1,8.7
2005-06-30,0.4,6.1
2005-09-30,0.4,6.0
2005-12-31,0.1,8.4


## Example 3: News Aggregator Dataset

Cada fila del dataset contiene el título, la URL, el nombre del medio que lo publicó y su dominio, además del timestamp de publicación. cluster es un ID aleatorio para el grupo de temas al que pertenece un artículo. category es la categoría de la noticia y contiene las siguientes opciones:

- b para negocios (business)
- t para ciencia y tecnología
- e para entretenimiento
- m para salud

In [28]:
df = pd.read_csv(
    "data/data_groupby/news.csv",
    sep="\t",
    header=None,
    index_col=0,
    names=["title", "url", "outlet", "category", "cluster", "host", "tstamp"],
    dtype={
        "outlet": "category",
        "category": "category",
        "cluster": "category",
        "host": "category",
    },
)

# Convierte la columna tstamp DESPUÉS de cargar el CSV
df["tstamp"] = pd.to_datetime(df["tstamp"], unit="ms")

df.iloc[0]

title       Fed official says weak data caused by weather,...
url         http://www.latimes.com/business/money/la-fi-mo...
outlet                                      Los Angeles Times
category                                                    b
cluster                         ddUyU0VZz0BRneMioxUPQVP6sIxvM
host                                          www.latimes.com
tstamp                             2014-03-10 16:52:50.698000
Name: 1, dtype: object

### Using Lambda Functions in .groupby()

Pregunta a responder: ¿Que outlet habla mas acerca de reserva federal?

Este código cuenta, para cada medio (outlet), cuántos títulos de artículos contienen la palabra "Fed" (refiriéndose a la Reserva Federal — "Federal Reserve"), y muestra los 10 medios con más menciones. df.groupby('outlet', sort=False) agrupa el DataFrame por medio de publicación, usando sort=False para evitar el ordenamiento automático por nombre de grupo. Sobre cada grupo, ['title'].apply(lambda ser: ser.str.contains('Fed').sum()) toma la columna title de ese grupo (una Serie con todos los títulos de ese medio) y aplica una función personalizada: ser.str.contains('Fed') genera una máscara booleana (True/False por cada título que contenga o no "Fed"), y .sum() cuenta cuántos True hay — es decir, cuántos títulos de ese medio mencionan "Fed". Finalmente, .nlargest(10) toma los 10 medios con el conteo más alto, ordenados de mayor a menor — mostrando qué medios cubrieron con más frecuencia noticias relacionadas con la Reserva Federal.

In [29]:
df.groupby('outlet', sort=False)['title'].apply(
    lambda ser: ser.str.contains('Fed').sum()
).nlargest(10)

outlet
Reuters                         161
NASDAQ                          103
Businessweek                     93
Investing.com                    66
Wall Street Journal \(blog\)     61
MarketWatch                      56
Moneynews                        55
Bloomberg                        53
GlobalPost                       51
Economic Times                   44
Name: title, dtype: int64

In [30]:
title, ser = next(iter(df.groupby('outlet', sort=False)['title']))
title

'Los Angeles Times'

In [31]:
ser.head()

1       Fed official says weak data caused by weather,...
486            Stocks fall on discouraging news from Asia
1124    Clues to Genghis Khan's rise, written in the r...
1146    Elephants distinguish human voices by sex, age...
1237    Honda splits Acura into its own division to re...
Name: title, dtype: str

### Improving the Performance of .groupby()

In [33]:
mentions_fed = df["title"].str.contains("Fed")
type(mentions_fed)

pandas.Series

1) mentions_fed.groupby(df['outlet'], sort=False) — agrupa la Serie booleana mentions_fed usando como criterio de agrupación la columna outlet del DataFrame original. Es una forma menos común pero válida de usar groupby(): en vez de llamarlo sobre el DataFrame completo, lo llamas sobre una Serie ya calculada, y le pasas otra Serie (df['outlet']) como el criterio de agrupación externo.
2) .sum() — suma los valores True/False dentro de cada grupo (recuerda: True vale 1, False vale 0), dando el conteo total de títulos que mencionan "Fed" por cada medio. Esto reemplaza el .apply(lambda ser: ser.str.contains('Fed').sum()) de la versión anterior — aquí el .str.contains('Fed') ya se calculó una sola vez de antemano (más eficiente si vas a reutilizar mentions_fed para otras cosas).
3) .nlargest(10) — igual que antes, toma los 10 medios con más menciones.
4) .astype(np.uintc) — convierte el resultado final a tipo entero sin signo (unsigned int, tipo C). Como los conteos siempre son números positivos (nunca negativos), este tipo ahorra memoria comparado con el int64 por defecto que usa pandas — un detalle de optimización, sin afectar el resultado.

In [34]:
import numpy as np
mentions_fed.groupby(df['outlet'], sort=False).sum().nlargest(10).astype(np.uintc)

outlet
Reuters                         161
NASDAQ                          103
Businessweek                     93
Investing.com                    66
Wall Street Journal \(blog\)     61
MarketWatch                      56
Moneynews                        55
Bloomberg                        53
GlobalPost                       51
Economic Times                   44
Name: title, dtype: uint32

In [36]:
import timeit

def test_apply():
    """Version 1: using `.apply()`"""
    df.groupby("outlet", sort=False)["title"].apply(
        lambda ser: ser.str.contains("Fed").sum()
    ).nlargest(10)

def test_vectorization():
    """Version 2: using vectorization"""
    mentions_fed = df["title"].str.contains("Fed")
    mentions_fed.groupby(
        df["outlet"], sort=False
    ).sum().nlargest(10).astype(np.uintc)

print(f"Version 1: {timeit.timeit(test_apply, number=3)}")
print(f"Version 2: {timeit.timeit(test_vectorization, number=3)}")

Version 1: 2.3865147999022156
Version 2: 0.2370220001321286


# pandas GroupBy: Putting It All Together

Si llamas `dir()` sobre un objeto GroupBy de pandas, vas a ver tantos métodos que puede marearte. Para ordenar mentalmente toda esa funcionalidad, conviene dividir los métodos en categorías según qué hacen y cómo se comportan:

1. **Métodos de agregación** (o de reducción) — combinan muchos datos en una sola estadística resumen. Ejemplo: tomar la suma, media o mediana de diez números, dando como resultado un solo número.

2. **Métodos de filtro** — devuelven un subconjunto del DataFrame original. Generalmente se usa `.filter()` para descartar grupos completos según alguna estadística comparativa de ese grupo. También entran aquí métodos que excluyen filas puntuales dentro de cada grupo.

3. **Métodos de transformación** — devuelven un DataFrame con la **misma forma e índices** que el original, pero con valores distintos. A diferencia de la agregación y el filtro (que suelen achicar el DataFrame), la transformación modifica los valores individuales pero conserva el tamaño original.

4. **Meta-métodos** — no se enfocan tanto en los datos originales, sino en darte información general de alto nivel, como la cantidad de grupos y sus índices.

5. **Métodos de graficado** — imitan la API de graficado normal de una Serie o DataFrame de pandas, pero típicamente dividen el resultado en varios subgráficos.

Hay algunos métodos de objetos GroupBy que no encajan bien en estas categorías — suelen producir un objeto intermedio que no es ni DataFrame ni Serie. Por ejemplo, `df.groupby().rolling()` produce un objeto `RollingGroupby`, sobre el cual después puedes llamar métodos de agregación, filtro o transformación.

Para profundizar más, la documentación de la API de `DataFrame.groupby()`, `DataFrame.resample()` y `pandas.Grouper` son buenos recursos.

# pivot_table(): tabla cruzada como alternativa a groupby()

"pivot_table()" es una forma alternativa a "groupby()" para resumir datos, pero organizando el resiltado como una **matriz** (categorias de una columna como filas, categoria de otra como columnas), en vez de una tabla 'larga'. Es mas visual cuando quieres comparar dos dimensiones al mismo tiempo.

In [38]:
dtypes = {
    "first_name": "category",
    "gender": "category",
    "type": "category",
    "state": "category",
    "party": "category",
}
df = pd.read_csv(
    "data/data_groupby/legislators-historical.csv",
    dtype=dtypes,
    usecols=list(dtypes) + ["birthday", "last_name"],
    parse_dates=["birthday"]
)
df.head()

,last_name,first_name,birthday,gender,type,state,party
0,Bassett,Richard,1745-04-02,M,sen,DE,Anti-Administration
1,Bland,Theodorick,1742-03-21,M,rep,VA,NaN
2,Burke,Aedanus,1743-06-16,M,rep,SC,NaN
3,Carroll,Daniel,1730-07-22,M,rep,MD,NaN
4,Clymer,George,1739-03-16,M,rep,PA,NaN


Utilizaremos "pivot_table()" para ver los legisladores por estado y genero.

In [39]:
df.pivot_table(
    index='state', 
    columns='gender', 
    values='last_name', 
    aggfunc='count', 
    fill_value=0
    ).head()

gender,F,M
state,,
AK,0,16
AL,3,203
AR,5,112
AS,0,2
AZ,3,45


`index="state"` define las filas, `columns="gender"` define las columnas, 
`values="last_name"` es la columna sobre la que se calcula la agregación 
(`aggfunc="count"` — cuenta cuántos legisladores hay en cada combinación), y 
`fill_value=0` reemplaza con 0 las combinaciones que no tengan datos (en vez de `NaN`).

In [40]:
# Equivalente con groupby()

df.groupby(['state', 'gender'])['last_name'].count().unstack(fill_value=0).head(10)

gender,F,M
state,,
AK,0,16
AL,3,203
AR,5,112
AS,0,2
AZ,3,45
CA,23,338
CO,3,87
CT,6,234
DC,0,2


Este `groupby()` + `.unstack()` produce el mismo resultado que `pivot_table()` — 
`.unstack()` es lo que convierte el índice jerárquico (state, gender) del `groupby()` 
en columnas separadas, imitando el formato de tabla cruzada. `pivot_table()` hace lo 
mismo en un solo paso, sin necesitar `.unstack()`.

# merge(): combinando tablas relacionadas

Para practicar `merge()`, creamos una tabla de referencia propia que relaciona cada 
estado con su región geográfica — simulando una relación "uno a muchos" típica de 
una base de datos relacional (una tabla de "hechos" y una tabla de "dimensión").

In [41]:
region_por_estado = pd.DataFrame({
    "state": ["CA", "OR", "WA", "NY", "NJ", "CT", "TX", "OK", "AR", "FL", "GA", "AL"],
    "region": ["Oeste", "Oeste", "Oeste", "Este", "Este", "Este",
               "Sur", "Sur", "Sur", "Sur", "Sur", "Sur"]
})

region_por_estado

,state,region
0,CA,Oeste
1,OR,Oeste
2,WA,Oeste
3,NY,Este
4,NJ,Este
5,CT,Este
6,TX,Sur
7,OK,Sur
8,AR,Sur
9,FL,Sur


"merge()" combina el dataframe de lagisladores con la tabla de regiones, usando "state" como la columna en comun (equivalente a un JOIN en SQL)

In [44]:
df_con_region = df.merge(
    region_por_estado,
    on='state',
    how='left'
    )

df_con_region[['last_name', 'state', 'region']].head()

,last_name,state,region
0,Bassett,DE,NaN
1,Bland,VA,NaN
2,Burke,SC,NaN
3,Carroll,MD,NaN
4,Clymer,PA,NaN


`on="state"` le dice a pandas qué columna usar para relacionar ambas tablas — 
equivalente a un `JOIN ... ON` en SQL. `how="left"` es el tipo de unión: conserva 
**todas** las filas de `df` (la tabla de la izquierda), y rellena con `NaN` la 
columna `region` en los legisladores de estados que no incluimos en nuestra tabla 
de referencia (ya que solo agregamos algunos estados de ejemplo).

Tipos de `how` disponibles, igual que en SQL:
- `"left"` → conserva todo de la tabla izquierda (equivalente a `LEFT JOIN`)
- `"right"` → conserva todo de la tabla derecha (equivalente a `RIGHT JOIN`)
- `"inner"` → solo las filas que coinciden en ambas tablas (equivalente a `INNER JOIN`)
- `"outer"` → conserva todo de ambas tablas (equivalente a `FULL OUTER JOIN`)

In [45]:
# Equivalente con groupby()

df_con_region.groupby('region')['last_name'].count().sort_values(ascending=False)

region
Este     2060
Sur      1135
Oeste     545
Name: last_name, dtype: int64

Esto demuestra el flujo típico de un caso de negocio real: **combinar** datos de 
distintas fuentes con `merge()`, y luego **resumir** el resultado con `groupby()` — 
justo lo que pide el objetivo del día.

## Repaso: 5 preguntas de negocio sobre el dataset de calidad del aire

In [51]:
df = pd.read_csv(
    "data/data_groupby/airqual.csv",
    na_values=[-200],
    usecols=["Date", "Time", "CO(GT)", "T", "RH", "AH"]
)

# Combina las columnas Date y Time en una sola columna datetime
df["Date_Time"] = pd.to_datetime(
    df["Date"] + " " + df["Time"], 
    dayfirst=True 
)

df = df.drop(columns=["Date", "Time"])

df = df.rename(
    columns={
        "CO(GT)": "co",
        "Date_Time": "tstamp",
        "T": "temp_c",
        "RH": "rel_hum",
        "AH": "abs_hum",
    }
).set_index("tstamp")

df.head()

C:\Users\nicol\AppData\Local\Temp\ipykernel_38660\3632527982.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date_Time"] = pd.to_datetime(


,co,temp_c,rel_hum,abs_hum
tstamp,,,,
2004-10-03 18:00:00,2.6,13.6,48.9,0.7578
2004-10-03 19:00:00,2.0,13.3,47.7,0.7255
2004-10-03 20:00:00,2.2,11.9,54.0,0.7502
2004-10-03 21:00:00,2.2,11.0,60.0,0.7867
2004-10-03 22:00:00,1.6,11.2,59.6,0.7888


### Pregunta 1: ¿Cuál es el promedio de CO por mes?

In [54]:
mensual_co = df.groupby(df.index.month)['co'].mean().sort_values(ascending=False)
mensual_co

tstamp
12    2.623177
11    2.560396
10    2.415101
4     2.387662
3     2.214482
9     2.206209
2     2.063141
1     2.007803
7     1.887222
5     1.861616
6     1.840529
8     1.674560
Name: co, dtype: float64

El mes con mayor promedio de monóxido de carbono es el mes 12 (Diciembre), probablemente asociado a mayor uso de 
calefacción o menor ventilación en esa época del año. El mes con menor promedio es 
8 (agosto), coincidiendo con temporada de mayor circulación de aire o menor 
combustión.

### Pregunta 2: ¿En qué categoría de temperatura (cool/warm/hot) la humedad relativa es más alta en promedio?

In [56]:
bins = pd.cut(df['temp_c'], bins=3, labels=('cool', 'warm', 'hot'))
df['rel_hum'].groupby(bins).agg(['mean', 'median'])

,mean,median
temp_c,,
cool,57.651452,59.2
warm,49.382716,49.3
hot,24.994334,24.1


La categoria "cool" tiene la humedad relativa mas alta. Esto tiene sentido ya que a menor temperatura el aire puede acumular mas humedad y tiene un punto de saturacion mas alto.

### Pregunta 3: ¿Cómo varía el promedio de CO según el día de la semana?

In [60]:
day_names = df.index.day_name()
dia_co = df.groupby(day_names)['co'].mean().sort_values(ascending=False)
dia_co

tstamp
Friday       2.435269
Wednesday    2.378685
Tuesday      2.327932
Thursday     2.273714
Monday       2.067529
Saturday     1.951165
Sunday       1.669290
Name: co, dtype: float64

El dia con mayor promedio de CO es viernes, mientras que el dia cpn menor promerio es domingo.

Probablemente relacionado con patrones de trafico vehicular entre dias laborales y fines de semana.

### Pregunta 4: ¿A qué hora del día se registra el pico más alto de CO en promedio?

In [67]:
hr = df.index.hour
df.groupby(hr)['co'].mean().rename_axis(['hr']).sort_values(ascending=False)

hr
19    3.733234
20    3.469069
18    3.436336
9     2.972477
8     2.823750
17    2.816314
21    2.600904
10    2.565749
16    2.267477
11    2.260923
13    2.201227
12    2.169632
14    2.126074
15    2.049231
22    1.976970
23    1.877508
7     1.810903
0     1.786018
1     1.467802
2     1.099063
6     0.921562
3     0.888462
4     0.758659
5     0.712934
Name: co, dtype: float64

El promedio mas alto de CO ocurre alrededor de las 19:00 horas, lo cual coincide con el trafico ya que las personas salen de sus trabajos y utilizan su auto para llegar a casa.

### Pregunta 5: ¿Cuál es el promedio mensual de temperatura y humedad absoluta a lo largo del período registrado?

In [70]:
resumen_mensual = df.resample('ME').agg({'temp_c':'mean', 'abs_hum':'mean'})
resumen_mensual

,temp_c,abs_hum
tstamp,,
2004-01-31,22.132547,1.400792
2004-02-29,23.225926,1.362804
2004-03-31,17.661756,0.966811
2004-04-30,19.131790,1.076649
2004-05-31,22.074924,1.008686
2004-06-30,25.250984,1.280975
2004-07-31,27.235224,1.217586
2004-08-31,26.269622,1.375433
2004-09-30,22.377778,1.263347


Se observa que la temperatura promedio baja, entre diciembre 2004 y enero 2005. La humedad absoluta es menor en 2005 en comparacion a 2004.

# Conclusión

En esta práctica profundicé en las herramientas de transformación y resumen de datos 
de pandas: `groupby()` con agregaciones múltiples y funciones personalizadas, 
`pivot_table()` como alternativa en formato de tabla cruzada, y `merge()` para 
combinar tablas relacionadas simulando una estructura tipo base de datos (equivalente 
a un `JOIN` de SQL), incluyendo los distintos tipos de unión (`left`, `right`, 
`inner`, `outer`).

Como cierre, apliqué estas técnicas para responder 5 preguntas de negocio sobre el 
dataset de calidad del aire de Italia (2004-2005). Los hallazgos principales: el 
monóxido de carbono es más alto en diciembre y más bajo en agosto, la humedad 
relativa es mayor en temperaturas frías, el viernes registra el promedio de CO más 
alto de la semana (probablemente por tráfico) y el domingo el más bajo, el pico 
diario de CO ocurre cerca de las 19:00 horas coincidiendo con el tráfico vespertino, 
y tanto la temperatura como la humedad absoluta bajan entre diciembre de 2004 y 
enero de 2005, siguiendo el patrón estacional esperado.

Este flujo de trabajo — agrupar, resumir, combinar tablas y responder preguntas de 
negocio con evidencia de los datos — es exactamente el tipo de análisis que se 
espera de un analista de datos junior en un caso real.